# 🏨 AI Travel Accommodation RAG System — LangChain Edition
### DEPI Graduation Project · Retrieval-Augmented Generation (LangChain)

---

## 🎯 Why this version differs from the manual notebook

Your earlier notebook (`Travel_RAG_System.ipynb`) implements RAG **manually** — calling
`sentence-transformers` and `chromadb` directly. This notebook implements the **identical
pipeline** using **LangChain's** abstractions instead, since that's what your instructor
specified.

| Concept | Manual version | This LangChain version |
|---|---|---|
| Chunking | Custom `hotel_to_chunk()` function | `RecursiveCharacterTextSplitter` |
| Documents | Plain Python dicts | LangChain `Document` objects |
| Embeddings | `SentenceTransformer` directly | `HuggingFaceEmbeddings` wrapper |
| Vector store | `chromadb.PersistentClient` directly | LangChain `Chroma` vectorstore |
| Retrieval | Manual `collection.query()` | `vectorstore.as_retriever()` |
| Generation | Manual prompt string + `generate_content()` | LCEL chain: `prompt \| llm \| parser` |
| LLM wrapper | Raw `google.generativeai` | `ChatGoogleGenerativeAI` (LangChain wrapper) |

The underlying concepts (scrape → chunk → embed → store → retrieve → generate) are
**identical**. Only the implementation framework changes.

## 🔑 Keys needed (same as before — reuse your existing secrets)
| Key | Where | Free tier |
|---|---|---|
| `GEMINI_API_KEY` | [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey) | ✅ |
| `SERPAPI_KEY` | [serpapi.com/manage-api-key](https://serpapi.com/manage-api-key) | ✅ |

## Cell 1 — Install Dependencies

In [ ]:
# ════════════════════════════════════════════════
# CELL 1 — Install Dependencies
# ════════════════════════════════════════════════
!pip install langchain langchain-community langchain-text-splitters langchain-google-genai langchain-chroma \
             sentence-transformers chromadb serpapi gradio --quiet
print('✅ Dependencies installed')

## Cell 2 — Imports & API Keys

In [ ]:
# ════════════════════════════════════════════════
# CELL 2 — Imports & API Keys
# ════════════════════════════════════════════════
import os
import re
from datetime import datetime, timedelta

import serpapi
from google.colab import userdata

# ── LangChain imports ─────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── Load keys from Colab Secrets ─────────────────────────────────
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
SERPAPI_KEY    = userdata.get('SERPAPI_KEY')

# LangChain's Gemini wrapper reads the key from this env var
os.environ['GOOGLE_API_KEY'] = GEMINI_API_KEY

print('✅ Imports done — LangChain + Gemini + SerpAPI ready')

## Cell 3 — Embeddings & Vector Store (LangChain wrappers)

In [ ]:
# ════════════════════════════════════════════════
# CELL 3 — Embeddings & Vector Store
# Uses LangChain's HuggingFaceEmbeddings + Chroma wrapper
# ════════════════════════════════════════════════
import time
from collections import deque

# ── Embedding model — same underlying model as the manual version,
#    wrapped in LangChain's standard embeddings interface ──────────
print('⏳ Loading embedding model (first run downloads ~80MB)...')
embedding_function = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

# ── LangChain's Chroma vectorstore wrapper ────────────────────────
# Persists to disk so the cache survives between cells/sessions
vectorstore = Chroma(
    collection_name='hotels_langchain',
    embedding_function=embedding_function,
    persist_directory='/content/hotel_vector_db_langchain'
)

# ── Simple in-memory cache tracker (city → last scraped timestamp) ─
# LangChain doesn't have a built-in cache-freshness primitive,
# so we track this ourselves, same as the manual version.
city_last_scraped = {}
REFRESH_DAYS = 14  # increased from 7 — fewer SerpAPI re-scrapes

# ── Simple token-bucket rate limiter for Gemini calls ─────────────
# Keeps us comfortably under the free-tier RPM ceiling (30 RPM for
# flash-lite) so we throttle/queue instead of hitting 429 errors.
class RateLimiter:
    def __init__(self, max_calls: int, period_seconds: float = 60.0):
        self.max_calls = max_calls
        self.period = period_seconds
        self.calls = deque()

    def wait(self):
        now = time.monotonic()
        while self.calls and now - self.calls[0] > self.period:
            self.calls.popleft()
        if len(self.calls) >= self.max_calls:
            sleep_time = self.period - (now - self.calls[0]) + 0.05
            if sleep_time > 0:
                print(f'⏳ Rate limit guard: waiting {sleep_time:.1f}s before next Gemini call...')
                time.sleep(sleep_time)
            now = time.monotonic()
            while self.calls and now - self.calls[0] > self.period:
                self.calls.popleft()
        self.calls.append(time.monotonic())

# Conservative cap — well under the published 30 RPM free-tier limit,
# leaving headroom since each user message can trigger up to 2 calls.
gemini_rate_limiter = RateLimiter(max_calls=12, period_seconds=60.0)

print('✅ HuggingFaceEmbeddings ready: all-MiniLM-L6-v2 (384 dimensions)')
print('✅ LangChain Chroma vectorstore ready at /content/hotel_vector_db_langchain')
print(f'✅ Cities refresh every {REFRESH_DAYS} days')
print(f'✅ Gemini rate limiter active: max {gemini_rate_limiter.max_calls} calls / {gemini_rate_limiter.period:.0f}s')


## Cell 4 — Scraping & LangChain Document Creation

In [ ]:
# ════════════════════════════════════════════════
# CELL 4 — Scraping + Building LangChain Documents
# ════════════════════════════════════════════════

def scrape_hotels_for_city(city: str, check_in: str = '2026-08-01', check_out: str = '2026-08-03') -> list:
    """
    Live scrape of hotel listings using SerpAPI Google Hotels.
    Identical scraping logic to the manual notebook — only the
    downstream document handling changes.
    """
    print(f'🌐 Scraping live hotel data for {city}...')
    try:
        serp_client = serpapi.Client(api_key=SERPAPI_KEY)
        results = serp_client.search({
            'engine'        : 'google_hotels',
            'q'             : f'Hotels in {city}',
            'check_in_date' : check_in,
            'check_out_date': check_out,
            'currency'      : 'USD',
            'hl'            : 'en',
            'gl'            : 'us'
        })
        properties = results.get('properties', [])
        if not properties:
            print(f'⚠️ No hotels found for {city}')
            return []

        hotels = []
        for h in properties[:20]:
            hotels.append({
                'name'           : h.get('name', 'Unknown'),
                'description'    : h.get('description', ''),
                'price_per_night': h.get('rate_per_night', {}).get('extracted_lowest', 0),
                'rating'         : h.get('overall_rating', 0.0),
                'reviews'        : h.get('reviews', 0),
                'amenities'      : h.get('amenities', []),
                'link'           : h.get('link', ''),
                'city'           : city
            })
        print(f'✅ Scraped {len(hotels)} hotels for {city}')
        return hotels

    except Exception as e:
        print(f'❌ Scraping failed for {city}: {e}')
        return []


def hotels_to_langchain_documents(hotels: list) -> list[Document]:
    """
    Convert raw hotel dicts into LangChain Document objects.
    Each Document has page_content (the text to embed) and
    metadata (structured fields used for filtering during retrieval).
    """
    documents = []
    for h in hotels:
        amenities_str = ', '.join(h['amenities']) if h['amenities'] else 'No listed amenities'
        content = (
            f"Hotel: {h['name']} in {h['city']}. "
            f"Price: ${h['price_per_night']} per night. "
            f"Rating: {h['rating']} out of 5 based on {h['reviews']} reviews. "
            f"Amenities: {amenities_str}. "
            f"Description: {h['description']}"
        )
        documents.append(Document(
            page_content=content,
            metadata={
                'city'           : h['city'].strip().lower(),
                'name'           : h['name'],
                'price_per_night': h['price_per_night'],
                'rating'         : h['rating'],
                'link'           : h['link']
            }
        ))
    return documents


# ── LangChain text splitter for chunking ──────────────────────────
# Each hotel listing is short, so this mostly passes documents through
# unchanged, but is included to satisfy the standard RAG chunking step
# and to handle any unusually long hotel descriptions safely.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', '']
)

print('✅ Scraping + LangChain Document builder ready')

## Cell 5 — Ingestion Pipeline (cache-aware, using LangChain Chroma)

In [ ]:
# ════════════════════════════════════════════════
# CELL 5 — Ingestion Pipeline
# ════════════════════════════════════════════════

def _normalize_city(city: str) -> str:
    return city.strip().lower()


def is_city_fresh(city: str) -> bool:
    """Check the in-memory cache timestamp for this city."""
    city_key = _normalize_city(city)
    if city_key not in city_last_scraped:
        return False
    age = datetime.now() - city_last_scraped[city_key]
    is_fresh = age < timedelta(days=REFRESH_DAYS)
    if is_fresh:
        print(f'✅ Cache HIT for {city} ({age.seconds // 3600}h old, fresh)')
    else:
        print(f'⏰ Cache STALE for {city}, refreshing')
    return is_fresh


def ingest_city(city: str) -> int:
    """
    Full LangChain ingestion pipeline:
      1. Scrape hotels
      2. Build LangChain Documents
      3. Split via RecursiveCharacterTextSplitter
      4. Add to LangChain Chroma vectorstore (embeds automatically)
    """
    city_key = _normalize_city(city)
    hotels = scrape_hotels_for_city(city)
    if not hotels:
        return 0

    # ── Remove old vectors for this city before re-adding ─────────
    try:
        existing = vectorstore.get(where={'city': city_key})
        if existing['ids']:
            vectorstore.delete(ids=existing['ids'])
            print(f'🗑️ Cleared {len(existing["ids"])} stale chunks for {city}')
    except Exception:
        pass

    # ── Build → split → embed → store, all via LangChain ──────────
    raw_documents = hotels_to_langchain_documents(hotels)
    split_documents = text_splitter.split_documents(raw_documents)

    vectorstore.add_documents(split_documents)  # embeds + stores in one call

    city_last_scraped[city_key] = datetime.now()
    print(f'✅ Ingested {len(hotels)} hotels ({len(split_documents)} chunks) for {city}')
    return len(hotels)


def ensure_city_ready(city: str) -> int:
    """Scrape only if missing or stale; otherwise serve from cache."""
    if is_city_fresh(city):
        existing = vectorstore.get(where={'city': _normalize_city(city)})
        return len(existing['ids'])
    return ingest_city(city)


print('✅ LangChain ingestion pipeline ready (7-day cache refresh)')

## Cell 6 — City Extraction (Gemini-based, reliable)

In [ ]:
# ════════════════════════════════════════════════
# CELL 6 — City Extraction
# Hybrid approach to minimize Gemini requests-per-minute:
#   1. Try a fast local keyword/regex match first (free, instant)
#   2. Only fall back to the Gemini LCEL chain if that fails
# ════════════════════════════════════════════════

city_extractor_llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash-lite',
    temperature=0
)

city_extraction_prompt = ChatPromptTemplate.from_messages([
    ('system', (
        'Extract ONLY the city name the user is asking about. '
        'Respond with JUST the city name, nothing else — no country, '
        'no punctuation, no explanation. If a country is mentioned but '
        'no specific city, respond with that country\'s capital city. '
        'If no location is mentioned, respond with exactly: NONE'
    )),
    ('human', '{message}')
])

# ── LCEL chain: prompt → llm → string output parser ───────────────
city_extraction_chain = city_extraction_prompt | city_extractor_llm | StrOutputParser()

_city_extract_cache = {}

# ── Lightweight local lookup so most messages never touch the LLM ──
# Covers common travel-query destinations; extend this list as needed.
KNOWN_CITIES = [
    'cairo', 'alexandria', 'giza', 'luxor', 'aswan', 'sharm el sheikh', 'hurghada',
    'paris', 'london', 'rome', 'milan', 'venice', 'florence', 'madrid', 'barcelona',
    'berlin', 'munich', 'amsterdam', 'vienna', 'prague', 'lisbon', 'athens',
    'istanbul', 'dubai', 'abu dhabi', 'doha', 'riyadh', 'amman', 'beirut',
    'tokyo', 'osaka', 'kyoto', 'seoul', 'beijing', 'shanghai', 'bangkok',
    'singapore', 'kuala lumpur', 'bali', 'jakarta', 'hong kong',
    'new york', 'los angeles', 'chicago', 'miami', 'san francisco', 'las vegas',
    'toronto', 'vancouver', 'mexico city', 'rio de janeiro', 'sao paulo',
    'sydney', 'melbourne', 'auckland', 'cape town', 'marrakech', 'casablanca'
]

import re as _re

def _local_extract_city(message: str) -> str | None:
    """Fast, free local match against known city names — no API call."""
    text = message.lower()
    for city in KNOWN_CITIES:
        if _re.search(r'\b' + _re.escape(city) + r'\b', text):
            return city.title()
    return None


def extract_city(message: str) -> str | None:
    """
    Extract a city name from a user message.
    Tries a free local match first; only calls the Gemini LCEL chain
    (rate-limited) when the local match fails — this is the main
    lever for keeping requests-per-minute low.
    """
    cache_key = message.strip().lower()
    if cache_key in _city_extract_cache:
        return _city_extract_cache[cache_key]

    # ── 1. Free local match ────────────────────────────────────────
    local_match = _local_extract_city(message)
    if local_match:
        _city_extract_cache[cache_key] = local_match
        return local_match

    # ── 2. Fall back to Gemini, throttled by the shared rate limiter ─
    try:
        gemini_rate_limiter.wait()
        city = city_extraction_chain.invoke({'message': message}).strip().strip('.')
        result = None if (not city or city.upper() == 'NONE' or len(city) > 50) else city
        _city_extract_cache[cache_key] = result
        return result
    except Exception as e:
        print(f'⚠️ City extraction failed: {e}')
        return None


print('✅ Hybrid city extraction ready (local match first, Gemini fallback + rate limit)')


## Cell 7 — RAG Chain (Retrieval + Generation via LCEL)

In [ ]:
# ════════════════════════════════════════════════
# CELL 7 — RAG Chain (LangChain Expression Language)
# This is the core retrieve-then-generate pipeline,
# built using LangChain's standard RAG pattern.
# ════════════════════════════════════════════════

answer_llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash-lite',
    temperature=0.3
)

RAG_PROMPT_TEMPLATE = ChatPromptTemplate.from_messages([
    ('system', (
        'You are a helpful travel accommodation assistant.\n\n'
        'Answer ONLY using the information in the CONTEXT below. '
        'Do not invent hotel names, prices, or amenities not present in the context. '
        "If the context doesn't contain enough information, say so clearly and "
        'suggest the user try a different city or relax their requirements. '
        'Mention specific hotel names, prices, and ratings when relevant. '
        'Be concise and friendly.\n\n'
        'CONTEXT:\n{context}'
    )),
    ('human', '{question}')
])


def format_retrieved_docs(docs: list[Document]) -> str:
    """Format retrieved LangChain Documents into a text block for the prompt."""
    if not docs:
        return 'No relevant hotels were found in the database for this query.'
    lines = []
    for i, doc in enumerate(docs, 1):
        link = doc.metadata.get('link', 'N/A')
        lines.append(f'[{i}] {doc.page_content} (link: {link})')
    return '\n'.join(lines)


def rag_answer(user_message: str, history: list) -> str:
    """
    Full LangChain RAG pipeline:
      1. Extract city (LCEL chain)
      2. Ensure city data is cached/fresh (scrape if needed)
      3. Build a city-scoped retriever and retrieve relevant chunks
      4. Run the RAG LCEL chain: context + question → Gemini → answer
    """
    try:
        city = extract_city(user_message)

        if city:
            hotel_count = ensure_city_ready(city)
            if hotel_count == 0:
                return (
                    f"⚠️ I couldn't find any hotel data for **{city}**. "
                    "This could be a spelling issue or limited availability. "
                    "Please try a different city name."
                )

        # ── Build a retriever, scoped to the city if one was found ──
        search_kwargs = {'k': 5}
        if city:
            search_kwargs['filter'] = {'city': _normalize_city(city)}

        retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)
        retrieved_docs = retriever.invoke(user_message)

        # ── LCEL RAG chain: retrieved context + question → answer ───
        rag_chain = (
            {
                'context' : lambda x: format_retrieved_docs(retrieved_docs),
                'question': RunnablePassthrough()
            }
            | RAG_PROMPT_TEMPLATE
            | answer_llm
            | StrOutputParser()
        )

        gemini_rate_limiter.wait()
        return rag_chain.invoke(user_message)

    except Exception as e:
        err = str(e)
        if '429' in err or 'quota' in err.lower():
            return '⚠️ **Rate limit hit.** Please wait a moment and try again.'
        if 'api_key' in err.lower() or 'invalid' in err.lower() or '401' in err:
            return (
                '⚠️ **API key issue.** Please check `GEMINI_API_KEY` and `SERPAPI_KEY` '
                'in Colab Secrets.'
            )
        return f'⚠️ **Unexpected error:** `{type(e).__name__}: {err}`'


print('✅ LangChain RAG chain ready (LCEL: retriever → prompt → llm → parser)')

## Cell 8 — Quick Test

In [ ]:
# ════════════════════════════════════════════════
# CELL 8 — Quick Test
# ════════════════════════════════════════════════
test_question = 'I need a luxury hotel in Paris with breakfast and a pool'
print(f'🧪 Testing LangChain RAG pipeline with: "{test_question}"\n')

answer = rag_answer(test_question, history=[])
print('━' * 60)
print('ANSWER:')
print(answer)

## Cell 9 — Gradio Chat UI

In [ ]:
# ════════════════════════════════════════════════
# CELL 9 — Gradio Chat UI
# ════════════════════════════════════════════════
import gradio as gr

with gr.Blocks(
    title='🏨 AI Travel RAG System (LangChain)',
    theme=gr.themes.Soft(primary_hue='blue')
) as demo:

    gr.Markdown("""
    # 🏨 AI Travel Accommodation RAG System — LangChain Edition
    ### DEPI Graduation Project · Built with LangChain (LCEL)
    Ask about hotels in any city — powered by a LangChain retrieval + generation chain.
    """)

    chatbot = gr.Chatbot(
        label='Travel RAG Assistant',
        type='messages',
        height=520,
        avatar_images=(None, 'https://img.icons8.com/fluency/48/hotel.png')
    )

    with gr.Row():
        msg = gr.Textbox(
            placeholder='e.g. Find me a luxury hotel in Rome with a pool',
            label='Your question',
            scale=4
        )
        send = gr.Button('Ask 🔍', variant='primary', scale=1)

    with gr.Row():
        clear = gr.Button('🗑️ Clear chat', scale=1, variant='secondary')

    gr.Examples(
        examples=[
            'Find me a hotel in Cairo under $100 per night',
            'I need a luxury hotel in Paris with breakfast and a pool',
            'Show me well-rated hotels in Tokyo',
            'Find budget hotels in Dubai under $80',
        ],
        inputs=msg
    )

    def respond(message: str, history: list):
        if not message.strip():
            return '', history
        reply = rag_answer(message, history)
        history.append({'role': 'user',      'content': message})
        history.append({'role': 'assistant', 'content': reply})
        return '', history

    send.click(respond,  [msg, chatbot], [msg, chatbot])
    msg.submit(respond,  [msg, chatbot], [msg, chatbot])
    clear.click(lambda: ([], ''), None, [chatbot, msg])

demo.launch(share=True, debug=True)

---
## 📋 LangChain Components Used

| Component | LangChain Class | Purpose |
|---|---|---|
| Chunking | `RecursiveCharacterTextSplitter` | Splits long text into overlapping chunks |
| Document model | `langchain_core.documents.Document` | Standard text + metadata container |
| Embeddings | `HuggingFaceEmbeddings` | Wraps sentence-transformers in LangChain's interface |
| Vector store | `langchain_chroma.Chroma` | Stores/retrieves embedded documents |
| Retriever | `vectorstore.as_retriever()` | Standard LangChain retrieval interface |
| LLM | `ChatGoogleGenerativeAI` | LangChain wrapper around Gemini |
| Prompting | `ChatPromptTemplate` | Structured, reusable prompt templates |
| Chain composition | LCEL (`prompt \| llm \| parser`) | LangChain Expression Language pipe syntax |
| Output parsing | `StrOutputParser` | Extracts plain text from the LLM response object |

---
*DEPI Graduation Project · RAG implemented with LangChain*